# Workshop IA Explicable (XAI)

> *« All models are wrong, but some are useful. »* (George E. P. Box, 1976)

**Pour démarrer :** exécutez les cellules de haut en bas avec **▶** (ou `Maj + Entrée`). Aucune installation requise.

In [ ]:
#@title Configuration et chargement des données (cliquez sur ▶)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
import statsmodels.iolib.summary as sm_sum


def _plain_forg(x, prec=3):
    x = float(np.squeeze(x))
    if np.isnan(x):
        return "      nan"
    if abs(x) >= 1e9 or (0 < abs(x) < 1e-4):
        return f"{x:9.3e}" if prec == 3 else f"{x:10.4e}"
    if prec == 4:
        return f"{x:10.2f}"
    return f"{x:9.2f}" if abs(x) >= 1000 else f"{x:9.3f}"


sm_sum.forg = _plain_forg

DATA_URL = "https://raw.githubusercontent.com/mickaeltemporao/workshop-xai/main/data/anes2024_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Données prêtes : {len(df)} foyers, {df.shape[1]} variables.")

## 1. Les données

Sur `4 734` foyers réels, nous cherchons si **faire des études est vraiment utile** pour expliquer le revenu annuel (`annual_income`), à partir de `6` critères : `education`, `age`, `employed`, `married`, `female` et `minority`.

In [ ]:
df.head()

In [ ]:
#@title Afficher la distribution des revenus (cliquez sur ▶)
mean_income = float(df["annual_income"].mean())
median_income = float(df["annual_income"].median())

fig, ax = plt.subplots(figsize=(8.6, 3.8))
ax.hist(df["annual_income"], bins=24, color="#0072bc", edgecolor="white", alpha=0.88)
ax.axvline(mean_income, color="#dd1019", linestyle="dashed", linewidth=2, label=f"Moyenne : {mean_income:,.0f} $ / an".replace(",", " "))
ax.axvline(median_income, color="#1e8449", linestyle="dotted", linewidth=2, label=f"Médiane : {median_income:,.0f} $ / an".replace(",", " "))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{int(x):,} $".replace(",", " ")))
ax.set_title("Distribution du revenu annuel (N = 4 734)")
ax.set_xlabel("Revenu annuel (annual_income, $ / an)")
ax.set_ylabel("Foyers")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Une droite (`2` paramètres)

Commençons par une régression linéaire **« boîte de verre »** avec un seul critère : les années d'études (`education`).

In [ ]:
model_simple = smf.ols("annual_income ~ education", data=df).fit()
print(model_simple.summary())

In [ ]:
#@title Afficher la droite de régression (cliquez sur ▶)
const_simple = float(model_simple.params["Intercept"])
slope_simple = float(model_simple.params["education"])

fig, ax = plt.subplots(figsize=(8.6, 4.3))
ax.scatter(df["education"], df["annual_income"], alpha=0.16, s=18, color="#0072bc", label="Foyers réels")
x_grid = np.linspace(df["education"].min(), df["education"].max(), 100)
ax.plot(
    x_grid,
    const_simple + slope_simple * x_grid,
    color="#dd1019",
    linewidth=2.4,
    label=f"Droite OLS : {const_simple:,.0f} + {slope_simple:,.0f} × education".replace(",", " "),
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f"{int(y):,} $".replace(",", " ")))
ax.set_title("Revenu annuel selon les années d'études")
ax.set_xlabel("Années d'études (education)")
ax.set_ylabel("Revenu annuel ($ / an)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Des paramètres (`7` paramètres)

Ajoutons les `5` autres variables pour mesurer l'effet de chaque levier **toutes choses égales par ailleurs**.

In [ ]:
model_full = smf.ols("annual_income ~ education + age + employed + married + female + minority", data=df).fit()
print(model_full.summary())

- La part expliquée (`R-squared`) passe de `0.178` à `0.313`, avec environ **`+10 998 $ / an`** par année d'études.
- À profil comparable, `female` (`-8 007 $ / an`) et `minority` (`-10 328 $ / an`) restent significatifs (`p < 0.001`) : un modèle appris sur des données historiques reproduit les inégalités existantes.

## 4. Hack-time : prédire un profil

Modifiez les valeurs ci-dessous et relancez la cellule pour observer la prédiction du modèle.

In [ ]:
my_profile = pd.DataFrame([{
    "education": 14,
    "age": 52,
    "employed": 1,
    "married": 1,
    "female": 0,
    "minority": 0,
}])

pred = model_full.predict(my_profile).iloc[0]
print(f"Revenu annuel prédit : {pred:,.0f} $ / an".replace(",", " "))

In [ ]:
#@title Afficher la contribution de chaque variable (cliquez sur ▶)
feature_names = ["education", "age", "employed", "married", "female", "minority"]
values = [
    float(model_full.params[f]) * (float(my_profile[f].iloc[0]) - float(df[f].mean()))
    for f in feature_names
]

fig, ax = plt.subplots(figsize=(8.6, 3.8))
colors = ["#0072bc" if v >= 0 else "#dd1019" for v in values]
bars = ax.barh(feature_names[::-1], values[::-1], color=colors[::-1], height=0.55)
ax.axvline(0.0, color="#333333", linewidth=1.1)
max_abs = max([abs(v) for v in values] + [15000.0])
ax.set_xlim(-max_abs * 1.45, max_abs * 1.45)
for bar, val in zip(bars, values[::-1]):
    ha = "left" if val >= 0 else "right"
    offset = max_abs * 0.03 if val >= 0 else -max_abs * 0.03
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2, f"{val:+,.0f} $".replace(",", " "), va="center", ha=ha, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{int(x):+,} $".replace(",", " ")))
ax.set_title("Écart de chaque variable par rapport au foyer moyen (bleu = +, rouge = -)")
plt.tight_layout()
plt.show()

### À vous de tester
1. Changez `"female": 0` en `"female": 1` (ou `"minority": 1`) dans `my_profile` pour vérifier l'écart exact (`-8 007 $` ou `-10 328 $`).
2. Testez `age = 30` puis `age = 70` dans l'**Onglet 4** de l'application web (`329` paramètres) pour comparer la courbe non linéaire au `-227 $ / an` fixe de la droite.

## 5. De la boîte de verre à la boîte noire

| Modèle | Paramètres | `R²` | Comment on l'explique |
| --- | ---: | ---: | --- |
| Régression simple | `2` | `0.18` | Lecture directe de la pente |
| Régression multiple | `7` | `0.31` | Lecture des `6` coefficients |
| Réseau de neurones (app web) | `329` | `0.34` | Sondes locales *post-hoc* (LIME, SHAP) |
| Grand modèle de langage (LLM) | `70 Md+` | - | Sondes ciblées et audit de circuits |

Dans une boîte noire, retirer `female` ou `minority` ne suffit pas toujours : d'autres variables (`married`, `education`, `employed`) peuvent agir comme **variables proxy** (comme dans le cas COMPAS). De plus, une corrélation ne prouve jamais une causalité.

*Données : [ANES 2024 Time Series Study](https://electionstudies.org/data-center/2024-time-series-study/) (`N = 4 734`). Supports préparés avec l'assistance d'outils d'IA générative, relus et validés par l'intervenant.*